<a href="https://colab.research.google.com/github/ytlLab/python/blob/main/ch11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#網頁request應用(JSON)

## Request
Request（請求） 是客戶端（通常是瀏覽器）與伺服器之間溝通的核心機制。

典型的請求遵循「請求－回應」（Request-Response）模型：

* Client（客戶端）：發起請求。例如：點擊按鈕或提交表單。

* HTTP Method（方法）：告訴伺服器要做什麼。
  * GET：獲取資料。
  * POST：新增資料。
  * PUT/PATCH：更新資料。
  * DELETE：刪除資料。清單項目
* Server（伺服器）：處理請求並回傳結果（Response）。

## JSON
早期的網頁中，資料交換常使用 XML，JSON (JavaScript Object Notation) 因為以下原因成為主流：

* 輕量化：比起 XML，JSON 的冗餘標籤極少，節省頻寬。
* 原生支持：JavaScript 可以直接將 JSON 轉換為物件（Object），不需要複雜的解析過程。
* 易讀性：結構清晰，人類與機器都能輕鬆閱讀。

## JSONPlaceholder


JSONPlaceholder 是一個免費的線上假資料 (Mock API) 服務，提供開發者測試、練習 Ajax、API 串接或製作原型 (Prototype) 而設計。

核心功能與特點：
* 免註冊免費使用： 無需 API 金鑰，直接即可使用。
* 豐富的假資料： 提供包含 Posts (文章)、Comments (留言)、Albums (相簿)、Photos (照片)、Todos (待辦事項)、Users (使用者) 等常見資料。
* 真實的 CRUD 操作： 例如使用 POST 建立資料，看起來會成功，但實際上不會真正將資料存入伺服器，非常適合前端測試流程。
* 支援 HTTPS： 確保在安全環境下進行數據請求。

In [ ]:
# 基礎 GET 請求：獲取全部貼文
# 使用 Python 的 requests 向伺服器發送請求，並接收 JSON 格式的資料

import requests
response = requests.get('https://jsonplaceholder.typicode.com/posts')
# 直接印出前兩筆原始資料
print(response.json()[:2])

In [ ]:
# 帶參數的 GET 請求：篩選資料
# 透過 params 篩選特定使用者

import requests
res = requests.get('https://jsonplaceholder.typicode.com/albums', params={'userId': 1})
print(res.json())

In [ ]:
# POST 請求：模擬新增資料
# 字典物件傳送至伺服器，伺服器會回傳包含新 ID 的完整物件。

import requests
payload = {'title': 'test title', 'body': 'test body', 'userId': 1}
res = requests.post('https://jsonplaceholder.typicode.com/posts', json=payload)
print(res.json())

In [ ]:
# PUT 請求：更新現有資料
# 針對特定 ID（例如 ID 1）進行資料覆蓋更新，並觀察回傳的更新後結果。
import requests
update_data = {'id': 1, 'title': 'new title', 'body': 'updated content', 'userId': 1}
res = requests.put('https://jsonplaceholder.typicode.com/posts/1', json=update_data)
print(res.json())

In [ ]:
# DELETE 請求：刪除資源
# 發送刪除請求。JSONPlaceholder 成功時通常會回傳一個空物件 {}。
import requests
res = requests.delete('https://jsonplaceholder.typicode.com/posts/1')
print("Status Code:", res.status_code)
print("Response:", res.json())

## 氣象資料開放平臺

需註冊取得API

https://opendata.cwa.gov.tw/index

不同的氣象資料有不同的ID（例如地震、雨量、自動觀測站），可以從官方平台的API文件查詢。

https://opendata.cwa.gov.tw/dist/opendata-swagger.html

In [ ]:
import requests
import pandas as pd

# --- 設定區 ---
# 請在此處輸入你的 API 授權碼
API_KEY = "你的授權碼"
# 範例資料 ID：F-C0032-001 (縣市預報資料)
DATA_ID = "F-C0032-001"
URL = f"https://opendata.cwa.gov.tw/api/v1/rest/datastore/{DATA_ID}"

# --- 發送請求 ---
params = {
    "Authorization": API_KEY,
    "format": "JSON",
    "locationName": "臺北市" # 可選填，不填則回傳全台灣
}

response = requests.get(URL, params=params)

# --- 資料解析 ---
if response.status_code == 200:
    data = response.json()

    # 提取縣市天氣資訊
    location = data['records']['location'][0]
    city_name = location['locationName']
    weather_elements = location['weatherElement']

    # 將 JSON 轉為易讀的列表
    weather_list = []
    for element in weather_elements:
        element_name = element['elementName']
        # 取得第一個時段的預報資料
        value = element['time'][0]['parameter']['parameterName']
        unit = element['time'][0]['parameter'].get('parameterUnit', '')
        weather_list.append({"項目": element_name, "數值": f"{value} {unit}"})

    # 轉換成 DataFrame 顯示
    df = pd.DataFrame(weather_list)
    print(f"--- {city_name} 目前天氣預報 ---")
    print(df)
else:
    print(f"請求失敗，錯誤代碼：{response.status_code}")

Colab 預設的字體不支援中文，需下載中文字體

(Google Noto Sans TC)

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os

# ==========================================
# 1. 環境設定：解決中文字體問題 (防止 0x55 錯誤)
# ==========================================
font_url = "https://github.com/googlefonts/noto-cjk/raw/main/Sans/OTF/TraditionalChinese/NotoSansCJKtc-Regular.otf"
font_name = "NotoSansTC.otf"

if not os.path.exists(font_name):
    print("正在下載中文字體，請稍候...")
    # 使用 -L 追蹤重新導向，確保下載到真實檔案
    !curl -L -o {font_name} {font_url}

# 檢查檔案是否損毀 (若小於 100KB 通常是下載到錯誤頁面)
if os.path.getsize(font_name) < 100000:
    print("⚠️ 字體下載異常，嘗試備用載點...")
    !curl -L -o {font_name} "https://github.com/google/fonts/raw/main/ofl/notosanstc/NotoSansTC-Regular.ttf"

font_prop = fm.FontProperties(fname=font_name)

# ==========================================
# 2. 設定區：請輸入你的 API KEY
# ==========================================
API_KEY = "你的授權碼"  # 👈 請在此處輸入你的氣象局 API 授權碼
CITY = "臺北市"         # 你想查詢的縣市
DATA_ID = "F-C0032-001"
URL = f"https://opendata.cwa.gov.tw/api/v1/rest/datastore/{DATA_ID}"

# ==========================================
# 3. 抓取資料並解析
# ==========================================
params = {
    "Authorization": API_KEY,
    "locationName": CITY,
    "format": "JSON"
}

try:
    response = requests.get(URL, params=params)
    response.raise_for_status() # 檢查 HTTP 狀態
    raw_data = response.json()

    # 提取縣市天氣元素
    location = raw_data['records']['location'][0]
    weather_elements = location['weatherElement']

    # 找出 MaxT (最高溫) 與 MinT (最低溫)
    max_t_data = [e for e in weather_elements if e['elementName'] == 'MaxT'][0]['time']
    min_t_data = [e for e in weather_elements if e['elementName'] == 'MinT'][0]['time']

    # 整理成 DataFrame
    time_labels = []
    max_temps = []
    min_temps = []

    for i in range(len(max_t_data)):
        # 格式化時間標籤 (例如 03/25 12時)
        t = max_t_data[i]['startTime']
        time_labels.append(f"{t[5:10]} {t[11:13]}時")
        max_temps.append(int(max_t_data[i]['parameter']['parameterName']))
        min_temps.append(int(min_t_data[i]['parameter']['parameterName']))

    print(f"成功取得 {CITY} 的氣象預報資料！")

except Exception as e:
    print(f"發生錯誤：{e}")
    # 若 API 失敗，以下繪圖將無法執行

# ==========================================
# 4. 繪製折線圖
# ==========================================
if 'max_temps' in locals():
    plt.figure(figsize=(12, 6), dpi=100)

    # 繪製數值線條
    plt.plot(time_labels, max_temps, marker='o', color='#e74c3c', label='預報最高溫', linewidth=2, markersize=8)
    plt.plot(time_labels, min_temps, marker='s', color='#3498db', label='預報最低溫', linewidth=2, markersize=8)

    # 加上數值標籤
    for i in range(len(time_labels)):
        plt.text(i, max_temps[i] + 0.5, f"{max_temps[i]}°C", ha='center', fontproperties=font_prop)
        plt.text(i, min_temps[i] - 1.5, f"{min_temps[i]}°C", ha='center', fontproperties=font_prop)

    # 美化圖表
    plt.title(f"{CITY} 未來 36 小時溫度趨勢", fontproperties=font_prop, fontsize=18, pad=20)
    plt.xlabel("預報時間段", fontproperties=font_prop, fontsize=12)
    plt.ylabel("溫度 (°C)", fontproperties=font_prop, fontsize=12)
    plt.legend(prop=font_prop, loc='upper right')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.ylim(min(min_temps)-3, max(max_temps)+3) # 自動調整 Y 軸間距

    plt.tight_layout()
    plt.show()

## 證交所 API

### 收盤成交資訊

In [ ]:
import requests
import pandas as pd
import time

# 設定 pandas 顯示完整欄位，方便觀察
pd.set_option('display.max_columns', None)

In [ ]:
def get_twse_v2(date_str):
    # API 網址
    url = f"https://www.twse.com.tw/exchangeReport/MI_INDEX?response=json&date={date_str}&type=ALL"
    headers = {"User-Agent": "Mozilla/5.0"}

    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        raw_data = response.json()

        # 1. 檢查有無資料
        if raw_data.get('stat') != 'OK':
            print(f"[{date_str}] 查無資料")
            return None

        # 2. 處理新版 'tables' 結構
        if 'tables' in raw_data:
            for table in raw_data['tables']:
                # 尋找包含「證券代號」的表格
                if "證券代號" in table.get('fields', []):
                    df = pd.DataFrame(table['data'], columns=table['fields'])
                    return df
            print("在 tables 中找不到個股行情")

        # 3. 備案：處理舊版 dataN 結構 (向下相容)
        else:
            for i in range(20):
                f_key = f'fields{i}'
                d_key = f'data{i}'
                if f_key in raw_data and "證券代號" in raw_data[f_key]:
                    return pd.DataFrame(raw_data[d_key], columns=raw_data[f_key])

    return None

# 測試執行
df_final = get_twse_v2("20260320") # 使用一個確定有開盤的日期測試
if df_final is not None:
    print("資料抓取成功！")
    display(df_final.head())

In [ ]:
print(df_final)

In [ ]:
import pandas as pd
from google.colab import files

# --- 1. 手動定義日期 (請確認與你爬取的日期一致) ---
date_str = "20260320"

# 假設 df 是你抓取到的個股行情資料（包含上千筆的那張表）
if 'df_final' in locals() and df_final is not None:

    # 設定檔名（建議加上日期）
    file_name = f"twse_data_{date_str}.csv"

    # 輸出成 CSV
    # index=False: 不儲存 pandas 的索引列
    # encoding='utf-8-sig': 這是重點！加上 -sig 才能讓 Excel 正常顯示中文不變亂碼
    df_final.to_csv(file_name, index=False, encoding='utf-8-sig')

    print(f"✅ 檔案 {file_name} 已生成。")

    # 自動觸發瀏覽器下載檔案
    files.download(file_name)
else:
    print("❌ 找不到資料表 df，請確保前面的抓取步驟已成功執行。")

## 其他
[12個有趣的API](https://codelove.tw/@tony/post/Ja6JaO)

## 作業1
使用氣象資料開放平臺爬取特定地區溫溼度、空氣品質等數據

##作業2
使用證交所API爬取資料後提取特定資料(如收盤價)繪出一週折線圖

## 補充 Pandas

Pandas 是一個開源的 Python 函式庫，專門用於數據操縱（Data Manipulation）與數據分析（Data Analysis）。它建立在 NumPy 之上，提供了一種高效能、易於使用的資料結構。

核心資料結構
Pandas 主要由兩個核心組件組成：

* Series：一維的資料結構，類似於列表或數組，但帶有索引（Index）。

* DataFrame：最常用的二維表格結構。可以想像成一個 Excel 工作表 或 SQL 表格，由多個 Series（列）組成。

In [ ]:
#基本操作範例
import pandas as pd

# 建立一個簡單的 DataFrame
data = {
    '姓名': ['小明', '小華', '小梅'],
    '數學': [90, 85, 95],
    '英文': [88, 92, 89]
}
df = pd.DataFrame(data)

# 計算數學的平均分
avg_math = df['數學'].mean()

# 篩選出數學大於 90 分的人
top_students = df[df['數學'] > 90]

print(avg_math)
print(top_students)

1.數據匯入與導出 (I/O)Pandas

支援多種檔案格式，最常用的是 CSV 和 Excel。
* 讀取數據：pd.read_csv('file.csv') 或 pd.read_excel('file.xlsx')
* 儲存數據：df.to_csv('output.csv', index=False)

2.數據檢視 (Inspection)

在拿到一份新資料時，這幾個指令能讓你快速了解資料的輪廓。
* df.head(n)：查看前 $n$ 筆資料（預設為 5 筆）。
* df.info()：查看資料欄位名稱、資料型態（Dtype）以及是否有缺失值（Non-null）。
* df.describe()：快速產出統計摘要（如平均值、標準差、最大最小值）。
* df.shape：查看資料的維度（幾列幾欄）。

3.數據選擇與過濾 (Selection & Filtering)

這是 Pandas 最靈活的部分。
* 選取特定欄位：df['欄位名稱'] 或 df[['欄位1', '欄位2']]
* 條件篩選：df[df['年齡'] > 20] (篩選出年齡大於 20 的所有列)。
* iloc / loc：
  * df.iloc[0:5, 0:2]：依據位置索引選取（第 0-4 列，第 0-1 欄）。
  * df.loc[:, '姓名']：依據標籤名稱選取。

4.數據清洗 (Cleaning)

原始資料通常很「髒」，需要處理缺失值或重複項。
* 處理缺失值：
  * df.isnull().sum()：計算每欄有多少空值。
  * df.dropna()：刪除包含空值的列。
  * df.fillna(value)：將空值填入特定數值。處理重複值：
* df.drop_duplicates()重新命名欄位：
* df.rename(columns={'舊名稱': '新名稱'})

5.數據聚合與分析 (Aggregation)

這是進行報表統計的核心。
* df.groupby('欄位').mean()：依照某欄位分組並計算平均。
* df.sort_values(by='欄位', ascending=False)：依照數值排序（由大到小）。
* df.apply(func)：對資料執行自定義的函式運算。